<a href="https://colab.research.google.com/github/InfoLang-Inc/llama-index-memory-infolang/blob/main/examples/InfoLangMemory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# InfoLang

[InfoLang](https://infolang.ai) is a hosted semantic memory API: `remember` stores text into a namespace, and `recall` retrieves the most relevant chunks for a query. `InfoLangMemory` adapts it to LlamaIndex's `BaseMemory` interface so agents and chat engines can persist and recall context across sessions.

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index llama-index-memory-infolang

### Setup

Set your InfoLang API key as an environment variable. You can replace `<your-infolang-api-key>` with your actual API key:

> Note: You can obtain your InfoLang API key from the [InfoLang dashboard](https://infolang.ai).

In [ ]:
import os

os.environ["INFOLANG_API_KEY"] = "il_live_..."

Create an `InfoLangMemory` instance. `namespace` scopes where memories are written and recalled from -- typically one namespace per user or per conversation thread.

In [ ]:
from llama_index.memory.infolang import InfoLangMemory

memory = InfoLangMemory.from_api_key(
    api_key=os.environ["INFOLANG_API_KEY"],
    namespace="user-1",
    top_k=5,  # optional, default is 5
)

You can also pass an already-constructed `infolang.InfoLang` client, e.g. to point at a self-hosted runtime:

In [ ]:
from infolang import InfoLang
from llama_index.memory.infolang import InfoLangMemory

client = InfoLang(dev_key="il_dev_...", base_url="http://127.0.0.1:8766")
memory = InfoLangMemory.from_client(client, namespace="user-1")

## Basic Usage

Currently, InfoLang Memory is supported in agents and chat engines.

Initialize the LLM

In [ ]:
import os
from llama_index.llms.openai import OpenAI

os.environ["OPENAI_API_KEY"] = "<your-openai-api-key>"
llm = OpenAI(model="gpt-4o")

### SimpleChatEngine

In [ ]:
from llama_index.core import SimpleChatEngine

chat_engine = SimpleChatEngine.from_defaults(
    llm=llm, memory=memory  # set your memory here
)

# Start the chat
response = chat_engine.chat("Hi, my name is Mayank")
print(response)

Later, in a new session with a fresh `InfoLangMemory` for the same namespace, InfoLang recalls the earlier turn automatically:

In [ ]:
new_session_memory = InfoLangMemory.from_client(client, namespace="user-1")
chat_engine = SimpleChatEngine.from_defaults(llm=llm, memory=new_session_memory)

response = chat_engine.chat("What's my name?")
print(response)

### FunctionAgent

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool


def call_fn(name: str):
    """Call the provided name.
    Args:
        name: str (Name of the person)
    """
    print(f"Calling... {name}")


call_tool = FunctionTool.from_defaults(fn=call_fn)

agent = FunctionAgent(
    tools=[call_tool],
    llm=llm,
)

# Start the chat
response = await agent.run("Hi, my name is Mayank", memory=memory)
print(response)

> Note: See the [repository README](https://github.com/InfoLang-Inc/llama-index-memory-infolang) for more usage details.

## References

- [InfoLang](https://infolang.ai)
- [InfoLang Python SDK](https://infolang.ai/docs/sdk/python)